# 12. Enumeration certificates (independent, about 2 minutes)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Singati2/LOYOLA-PAPER/blob/review/match-figure-2026-10-20/colab/12_enumeration_certificates.ipynb)

Re-runs the four scripts of `audit/certificates/`, written independently of the repository's verifiers: the minimal orders 13 (all trees) and 16 (maximum degree at most 4) at which distinct degree-pair profiles share a degree-ratio histogram, the first order for each bound on the maximum degree, the first orders at which SDD and ISDD merge distinct profiles (9 and 10), and the witness trees with the pendant-path construction. Exact rational arithmetic; only `networkx` is needed. No NIST data are used, and no Drive folder or state folder is involved, so this notebook can be run on its own, in any order.

In [ ]:
# Setup: check out the pinned commit and install networkx at the version that produced the committed outputs
import base64, hashlib, json, os, shutil, subprocess, sys


def git_net(args, token=None, cwd=None):
    """Run a git command that may need the network; on failure raise with the token redacted."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    secrets = []
    if token:
        auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
        secrets = [token, auth]
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0="Authorization: Basic " + auth)
    try:
        r = subprocess.run(["git", *args], cwd=cwd, env=env, capture_output=True, text=True)
        msg = (r.stdout + r.stderr).strip()
        for s in secrets:
            msg = msg.replace(s, "***")
    finally:
        env = secrets = auth = None
    if r.returncode != 0:
        raise RuntimeError("git " + args[0] + " failed: " + msg[-800:])


def at_pin(work, pin, fetch_args, token=None):
    """Make the checkout at work be exactly the commit pin, with no local changes, or stop."""
    git = lambda *a: subprocess.run(["git", "-C", work, *a], capture_output=True, text=True)
    if git("status", "--porcelain", "--untracked-files=no").stdout.strip():
        raise RuntimeError(work + " has local changes to tracked files; delete that folder and run this cell again")
    if git("rev-parse", "HEAD").stdout.strip() != pin:
        if git("cat-file", "-e", pin + "^{commit}").returncode != 0:
            git_net(["fetch", "--quiet", *fetch_args], token, cwd=work)
        r = git("checkout", "--quiet", "--detach", pin)
        if r.returncode != 0:
            raise RuntimeError("cannot check out the pinned commit " + pin + ": " + r.stderr.strip())
    head = git("rev-parse", "HEAD").stdout.strip()
    if head != pin:
        raise RuntimeError("checkout is at " + head + ", not the pinned commit " + pin)
    return head

REVISION = "review/match-figure-2026-10-20"   # the branch or tag that is cloned; it must contain PIN
PIN = os.environ.get("LOYOLA_PIN", "b7f50c094c815e25cbce356f878de89814a0395a")   # the exact commit this notebook runs
if not os.path.exists("minimal.py"):
    while not os.path.exists("verify.py") and os.getcwd() != "/":   # inside a local checkout: go to its top folder
        os.chdir("..")
    if not os.path.exists("verify.py"):                             # on Colab: clone the revision
        os.chdir("/content" if os.path.isdir("/content") else os.path.expanduser("~"))
        if not os.path.exists("LOYOLA-PAPER"):
            git_net(["clone", "--quiet", "--branch", REVISION, "https://github.com/Singati2/LOYOLA-PAPER.git", "LOYOLA-PAPER"])
        os.chdir("LOYOLA-PAPER")
    at_pin(os.getcwd(), PIN, ["https://github.com/Singati2/LOYOLA-PAPER.git", REVISION])
    os.chdir("audit/certificates")
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "networkx==3.6.1"], check=True)
import networkx
print("python", sys.version.split()[0], "| networkx", networkx.__version__)
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
def run(*args):
    """Run one certificate script, print its last line, return its full output."""
    out = subprocess.run([sys.executable, *args], capture_output=True, text=True, check=True).stdout
    print(args[0], "->", out.strip().splitlines()[-1])
    return out

a = run("minimal.py", "17")
assert a.strip().splitlines()[-1] == "{'all': 13, 'chem': 16}"
b = run("by_delta.py", "18")
assert "'4': 16, '5': 16" in b and "'None': 13" in b and "'3'" not in b.splitlines()[-1]
c = run("ratio_indices.py")
assert "('SDD', 'chem'): 9" in c and "('ISDD', 'chem'): 10" in c
d = run("witnesses.py")
assert d.count("OK for every order") == 2
print("ALL CERTIFICATES REPRODUCED")